# LLM & RAG Pipeline




## 0. Setup

In [4]:
%pip install -q faiss-cpu sentence-transformers rank-bm25 \
                 torch transformers \
                 langchain-openai langchain-core \
                 requests python-dotenv

In [5]:
import os
import sys
import json
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from google.colab import userdata

# Loads OPENROUTER_API_KEY / GEMINI_API_KEY / ADMIN_PASSWORD from a .env file
# in the project root, per the README's "Setup & Installation" step 3.
load_dotenv()

# Make sure the sibling P2/P3 modules (copied next to this notebook) are importable.
sys.path.append(os.getcwd())

print("OPENROUTER_API_KEY set:", bool(userdata.get("OPEN_ROUTER_KEY")))
print("GEMINI_API_KEY set:", bool(userdata.get("GEMINI_API_KEY")))

OPENROUTER_API_KEY set: True
GEMINI_API_KEY set: True


In [6]:
# P2 — Search & RAG Engineer's retrieval engine (imported unmodified)
from p2_retrieval_engine import SalesRetrievalEngine, _mock_products


## 1. LLM Manager — Dual-API (OpenRouter → Gemini → static fallback)


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import requests


class LLMManager:
    """Dual-API LLM caller: OpenRouter (primary) -> Gemini (backup) -> static fallback."""

    FALLBACK_MESSAGE = (
        "I'm sorry, I'm having temporary technical issues. "
        "Please try again in a moment."
    )

    def __init__(
        self,
        openrouter_model: str = None,
        gemini_model: str = "gemini-2.5-flash",
        temperature: float = 0.2,
        timeout: int = 10,
    ) -> None:
        self.openrouter_model = openrouter_model or os.getenv(
            "OPENROUTER_MODEL", "cohere/north-mini-code:free"
        )
        self.gemini_model = gemini_model
        self.gemini_key = userdata.get("GEMINI_API_KEY")
        self.gemini_url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/"
            f"{self.gemini_model}:generateContent"
        )

        # Constructing ChatOpenAI never makes a network call, so this is safe
        # even when OPENROUTER_API_KEY is missing -- the failure (if any)
        # only happens later, inside .invoke(), and is caught in chat().
        self.primary_llm = ChatOpenAI(
            model=self.openrouter_model,
            openai_api_key=userdata.get("OPEN_ROUTER_KEY"),
            openai_api_base="https://openrouter.ai/api/v1",
            temperature=temperature,
            max_retries=1,
            timeout=timeout,
        )

        # Records who answered each turn: "openrouter" | "gemini_backup" | "none"
        self.usage_log: List[Dict[str, str]] = []

    def chat(self, system_prompt: str, user_prompt: str) -> Dict[str, str]:
        """Try OpenRouter, then Gemini, then a static fallback. Never raises."""

        # ── Attempt 1: OpenRouter (primary) ────────────────────────────
        try:
            messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            response = self.primary_llm.invoke(messages)
            self.usage_log.append({"source": "openrouter", "status": "success"})
            return {"content": response.content, "source": "openrouter", "status": "success"}
        except Exception as e:
            print(f"[LLMManager] OpenRouter failed: {e}")

            # ── Attempt 2: Gemini (backup, free REST API) ──────────────
            try:
                text = self._call_gemini(system_prompt, user_prompt)
                self.usage_log.append({"source": "gemini_backup", "status": "fallback"})
                return {"content": text, "source": "gemini_backup", "status": "fallback"}
            except Exception as e2:
                print(f"[LLMManager] Gemini also failed: {e2}")

                # ── Attempt 3: static fallback ──────────────────────────
                self.usage_log.append({"source": "none", "status": "failed"})
                return {"content": self.FALLBACK_MESSAGE, "source": "none", "status": "failed"}

    def _call_gemini(self, system_prompt: str, user_prompt: str) -> str:
        if not self.gemini_key:
            raise RuntimeError("GEMINI_API_KEY is not set.")
        url = f"{self.gemini_url}?key={self.gemini_key}"
        payload = {
            "contents": [{"parts": [{"text": f"{system_prompt}\n\nUser: {user_prompt}"}]}],
            "generationConfig": {"temperature": 0.2, "maxOutputTokens": 500},
        }
        r = requests.post(url, json=payload, timeout=15)
        r.raise_for_status()
        data = r.json()
        return data["candidates"][0]["content"]["parts"][0]["text"]


## 2. Prompt Builder — the "A" in RAG


In [8]:
class PromptBuilder:
    """Builds the grounded system and user prompt sent to the LLM."""

    SYSTEM_PROMPT = """You are SmartSales Bot, a friendly, professional sales assistant for an e-commerce store.

Rules:
1. Use ONLY the provided product data for product facts, availability, prices, and ratings. Never invent them.
2. If the provided data is insufficient, say "I don't have that information."
3. Keep responses concise, helpful, and professional.
4. Include prices and ratings when mentioning products.
5. When the user describes a support problem, acknowledge it clearly and offer appropriate next steps.
6. Show a supplied installment plan accurately; do not invent plan terms.
7. Recommend related products only when they are supported by the retrieved product data.
8. Reply in the same language the user wrote in (Arabic or English)."""

    def build_prompt(
        self,
        user_message: str,
        products_found: List[Dict[str, Any]],
        viewed_products: List[Dict[str, Any]],
        chat_history: List[Dict[str, str]],
        installment_info: Optional[Dict[str, Any]] = None,
    ) -> str:
        return f"""
===============================================
USER MESSAGE: {user_message}
===============================================

POTENTIALLY RELEVANT PRODUCTS FROM SEARCH:
===============================================
{self._format_products(products_found)}

===============================================
INSTALLMENT PLAN (only when supplied):
===============================================
{self._format_installment(installment_info)}

===============================================
PRODUCTS USER ALREADY VIEWED:
===============================================
{self._format_viewed(viewed_products)}

===============================================
CONVERSATION HISTORY (last 5 messages):
===============================================
{self._format_history(chat_history)}

===============================================
INSTRUCTIONS:
===============================================
Answer the user's message directly. Use a retrieved product only when it is relevant to the
question. State exact prices and ratings only from the supplied product data. If an installment
plan is supplied, show its terms accurately. If product data does not answer the question, say
what information is unavailable rather than making up a product fact.
"""

    def _format_products(self, products: List[Dict[str, Any]]) -> str:
        if not products:
            return "No products found."
        lines = []
        for i, p in enumerate(products, 1):
            price = p.get("final_price", "N/A")
            initial = p.get("initial_price")
            discount = p.get("discount")
            price_line = f"   Price: ${price}"
            if initial and discount:
                price_line += f" (was ${initial}, {discount}% off)"
            lines.append(
                f"{i}. {p.get('title', 'Unknown product')}\n"
                f"{price_line}\n"
                f"   Rating: {p.get('rating', 'N/A')}/5 ({p.get('ratings_count', 0)} reviews)\n"
                f"   Category: {p.get('category', 'N/A')}\n"
                f"   Customers said: {p.get('what_customers_said', 'N/A')}"
            )
        return "\n\n".join(lines)

    def _format_installment(self, installment_info: Optional[Dict[str, Any]]) -> str:
        if not installment_info:
            return "No installment plan was supplied."
        return (
            f"{installment_info['months']} months -> "
            f"${installment_info['monthly_payment']}/month "
            f"(total ${installment_info['total_with_interest']})"
        )

    def _format_viewed(self, viewed: List[Dict[str, Any]]) -> str:
        if not viewed:
            return "User hasn't viewed any products yet."
        return ", ".join(p.get("title", "Unknown") for p in viewed)

    def _format_history(self, history: List[Dict[str, str]]) -> str:
        if not history:
            return "This is the start of the conversation."
        lines = []
        for msg in history[-5:]:
            role = "User" if msg.get("role") == "user" else "Bot"
            lines.append(f"{role}: {msg.get('content', '')}")
        return "\n".join(lines)


## 3. Conversation Memory (minimal stand-in for P5)

P5 (*Business Logic & Memory Engineer*) owns the real persistent memory layer per the README.
Only P2's work was provided here, so this is a small in-memory placeholder — just enough state
(chat history and last-viewed products, per `session_id`) to exercise `RAGPipeline` end-to-end in
this notebook. Swap it for P5's real class later; `RAGPipeline` only needs the methods below.


In [9]:
class ConversationMemory:
    """In-memory placeholder for P5's persistent memory/session store."""

    def __init__(self) -> None:
        self._sessions: Dict[str, Dict[str, Any]] = {}

    def _session(self, session_id: str) -> Dict[str, Any]:
        return self._sessions.setdefault(session_id, {"messages": [], "viewed_products": []})

    def save_message(self, session_id: str, role: str, content: str) -> None:
        self._session(session_id)["messages"].append({"role": role, "content": content})

    def get_last_n_messages(self, session_id: str, n: int = 5) -> List[Dict[str, str]]:
        return self._session(session_id)["messages"][-n:]

    def get_all_messages(self, session_id: str) -> List[Dict[str, str]]:
        return self._session(session_id)["messages"]

    def add_viewed_products(self, session_id: str, products: List[Dict[str, Any]]) -> None:
        seen_ids = {p.get("product_id") for p in self._session(session_id)["viewed_products"]}
        for p in products:
            if p.get("product_id") not in seen_ids:
                self._session(session_id)["viewed_products"].append(p)
                seen_ids.add(p.get("product_id"))

    def get_viewed_products(self, session_id: str) -> List[Dict[str, Any]]:
        return self._session(session_id)["viewed_products"]

    def get_last_viewed_product(self, session_id: str) -> Optional[Dict[str, Any]]:
        viewed = self._session(session_id)["viewed_products"]
        return viewed[-1] if viewed else None


## 4. Explicit human-handoff policy

Human handoff no longer depends on any classifier or score. `HumanHandoffPolicy` has two
independent routes:

1. The frontend can set `request_handoff=True`, for example from a **Contact a representative**
   control.
2. The customer directly asks to speak with a person using the supported English or Arabic phrases.

This keeps escalation deterministic and auditable. It avoids inferring whether a customer should
be transferred from their language or tone. The policy returns a reason code for the frontend and
for the agent summary.


In [10]:
class HumanHandoffPolicy:
    """Route explicit requests for a human representative without model scoring."""

    _EXPLICIT_REQUEST_PHRASES = (
        "talk to a human",
        "speak to a human",
        "talk to a person",
        "speak to a person",
        "human agent",
        "live agent",
        "human representative",
        "customer representative",
        "contact support",
        "customer service",
        "موظف خدمة العملاء",
        "دعم بشري",
        "اتكلم مع حد",
        "اتكلم مع شخص",
        "اكلم حد",
        "أكلم حد",
        "أتكلم مع حد",
        "موظف",
    )

    def evaluate(self, user_message: str, request_handoff: bool = False) -> Optional[str]:
        """Return a reason code when an explicit handoff route is requested."""
        if request_handoff:
            return "frontend_request"

        normalized = " ".join((user_message or "").casefold().split())
        if any(phrase in normalized for phrase in self._EXPLICIT_REQUEST_PHRASES):
            return "explicit_customer_request"
        return None


## 5. RAG Pipeline — wiring P2 and P4 together

`process_message()` is the single entry point that `app.py` (Gradio) would call for each user turn.
It:

1. Applies the deterministic `HumanHandoffPolicy` before generating a reply.
2. Retrieves potentially relevant products through `engine.hybrid_search()`.
3. Optionally calculates a plan only when the frontend explicitly supplies `installment_months`.
4. Builds the grounded P4 prompt and calls the LLM.
5. Saves the turn to memory and returns a structured result.

The pipeline has no dependency on model-derived language analysis or legacy automatic escalation behavior.


In [11]:
class RAGPipeline:
    """Orchestrates P2 retrieval and P4 prompt generation, memory, handoff, and LLM calls."""

    def __init__(
        self,
        engine: SalesRetrievalEngine,
        llm_manager: Optional[LLMManager] = None,
        prompt_builder: Optional[PromptBuilder] = None,
        memory: Optional[ConversationMemory] = None,
        handoff_policy: Optional[HumanHandoffPolicy] = None,
    ) -> None:
        self.engine = engine
        self.llm = llm_manager or LLMManager()
        self.prompt_builder = prompt_builder or PromptBuilder()
        self.memory = memory or ConversationMemory()
        self.handoff_policy = handoff_policy or HumanHandoffPolicy()

    def process_message(
        self,
        user_message: str,
        session_id: str,
        request_handoff: bool = False,
        installment_months: Optional[int] = None,
    ) -> Dict[str, Any]:
        """Handle one message without classifier-derived fields or dependencies.

        `request_handoff` is a frontend-controlled explicit action. `installment_months` is also
        frontend-controlled, allowing a UI to request a calculation for the customer's last viewed
        product without interpreting the customer's message in this pipeline.
        """
        handoff_reason = self.handoff_policy.evaluate(user_message, request_handoff)
        self.memory.save_message(session_id, "user", user_message)

        if handoff_reason:
            summary = self._generate_handoff_summary(session_id, handoff_reason)
            return {
                "type": "handoff",
                "message": "I’ll connect you with a human representative.",
                "summary_for_agent": summary,
                "handoff_reason": handoff_reason,
            }

        products = self._retrieve_products(user_message)
        installment_info = self._build_installment_plan(session_id, installment_months)

        chat_history = self.memory.get_last_n_messages(session_id, n=5)
        viewed_products = self.memory.get_viewed_products(session_id)
        prompt = self.prompt_builder.build_prompt(
            user_message=user_message,
            products_found=products,
            viewed_products=viewed_products,
            chat_history=chat_history,
            installment_info=installment_info,
        )

        llm_response = self.llm.chat(
            system_prompt=self.prompt_builder.SYSTEM_PROMPT,
            user_prompt=prompt,
        )

        self.memory.save_message(session_id, "bot", llm_response["content"])
        if products:
            self.memory.add_viewed_products(session_id, products[:1])

        return {
            "type": "response",
            "message": llm_response["content"],
            "products": products,
            "installment": installment_info,
            "llm_source": llm_response["source"],
        }

    def _retrieve_products(self, user_message: str) -> List[Dict[str, Any]]:
        """Retrieve evidence for every non-empty message; an empty message has no search query."""
        if not (user_message or "").strip():
            return []
        try:
            return self.engine.hybrid_search(user_message, top_k=5)
        except Exception as error:
            print(f"[RAGPipeline] Retrieval failed: {error}")
            return []

    def _build_installment_plan(
        self, session_id: str, installment_months: Optional[int]
    ) -> Optional[Dict[str, Any]]:
        """Calculate a plan only when the frontend explicitly supplies valid month count."""
        if (
            not isinstance(installment_months, int)
            or isinstance(installment_months, bool)
            or installment_months <= 0
        ):
            return None

        viewed = self.memory.get_last_viewed_product(session_id)
        if not viewed or not viewed.get("final_price"):
            return None
        return self.engine.calculate_installment(
            float(viewed["final_price"]), months=installment_months
        )

    def _generate_handoff_summary(self, session_id: str, handoff_reason: str) -> str:
        history = self.memory.get_all_messages(session_id)
        summary_prompt = (
            "Summarize this customer conversation for a human agent. "
            "Include the latest request, products viewed, any unresolved support issue, and the "
            f"handoff reason code ({handoff_reason}). Be concise and factual.\n\n"
            f"Conversation:\n{history}"
        )
        result = self.llm.chat(
            "You are a conversation summarizer. Be concise and factual.",
            summary_prompt,
        )
        return result["content"]


## 6. Demo run

Uses P2's own mock catalogue (`_mock_products()`, 4 EN/AR laptops plus a monitor), so the whole
notebook is runnable standalone without the real SQLite/`data_pipeline` database. Swap in
`engine.build_indexes_from_database(...)` (see `P2_Work_Main.py`) to run against the real
913-product catalogue instead.

If `OPENROUTER_API_KEY` and `GEMINI_API_KEY` are not set, `llm_source` will show `"none"` and
`message` will be the static fallback. That is expected offline; retrieval, prompt, memory, and
explicit handoff wiring still run normally.


In [12]:
engine = SalesRetrievalEngine()
engine.build_indexes(_mock_products())

pipeline = RAGPipeline(engine)
session_id = "demo-session-1"

conversation = [
    {"message": "Hi, show me a cheap gaming laptop"},
    {"message": "What's the price of the ASUS one?"},
    {"message": "Can I pay in installments?", "installment_months": 6},
    {"message": "This is broken, I want to talk to a human"},
]

for turn in conversation:
    result = pipeline.process_message(
        turn["message"],
        session_id=session_id,
        request_handoff=turn.get("request_handoff", False),
        installment_months=turn.get("installment_months"),
    )
    print("=" * 70)
    print("USER:", turn["message"])
    print("-" * 70)
    print(
        f"type={result['type']}  "
        f"handoff_reason={result.get('handoff_reason', 'n/a')}  "
        f"llm_source={result.get('llm_source', 'n/a')}"
    )
    print("-" * 70)
    print("BOT:", result["message"])
    if result.get("products"):
        print("\nProducts surfaced:", [p["title"] for p in result["products"]])
    if result.get("installment"):
        print("Installment plan:", result["installment"])
    if result.get("summary_for_agent"):
        print("\nHandoff summary for agent:", result["summary_for_agent"])
    print()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

USER: Hi, show me a cheap gaming laptop
----------------------------------------------------------------------
type=response  handoff_reason=n/a  llm_source=openrouter
----------------------------------------------------------------------
BOT: Here are the gaming laptops available:

**Lenovo Legion 5 Gaming Laptop**  
- Price: **$1,049**  
- Rating: **4.4/5** (1,240 reviews)  

**ASUS ROG Strix G‑15 Gaming Laptop**  
- Price: **$1,199**  
- Rating: **4.6/5** (812 reviews)  

The Lenovo Legion 5 is the more budget‑friendly option at $1,049. Let me know if you’d like more details or help with anything else!

Products surfaced: ['ASUS ROG Strix G-15 Gaming Laptop', 'Lenovo Legion 5 Gaming Laptop', 'Dell XPS 13 Ultrabook', 'شاشة سامسونج 27 بوصة للألعاب']

USER: What's the price of the ASUS one?
----------------------------------------------------------------------
type=response  handoff_reason=n/a  llm_source=openrouter
----------------------------------------------------------------------

In [13]:
# LLM usage log — which API answered each turn (useful for an ops/admin dashboard)
pipeline.llm.usage_log


[{'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'}]